In [166]:
import wandb
import pandas as pd

from matplotlib import pyplot as plt

In [167]:
pd.set_option('display.max_rows', None)

In [168]:
import json

In [169]:
api = wandb.Api(timeout=120)

In [170]:
prj = "gmum/bloraxs2"
runs = api.runs(prj,)

In [171]:
def extract_wandb_params(config_dict):
    """
    Extracts task, reconstruction type, seed, and start time from W&B config.
    """
    def get_val(key):
        return config_dict.get(key, {}).get('value')

    # 1. Extract basic params
    extracted = {
        "task_name": get_val("experiment/task") or get_val("task"),
        "reconstruction_type": get_val("reconstruction_type"),
        "seed": get_val("experiment/seed") or get_val("seed"),
        "start_time": None
    }

    # 2. Extract Date/Time from the nested _wandb metadata
    # Path: _wandb -> value -> e -> [dynamic_hash_key] -> startedAt
    wandb_metadata = config_dict.get("_wandb", {}).get("value", {}).get("e", {})
    
    for run_id in wandb_metadata:
        start_str = wandb_metadata[run_id].get("startedAt")
        if start_str:
            # Returns the ISO string: "2026-01-15T12:09:44.079821Z"
            extracted["start_time"] = start_str
            break

    return extracted

In [172]:
def extract_stats(df, dirs = ["map_metrics/", "laplace_LORAXS_KRON/", "laplace_LORAXS_DIAG/", "laplace/base_"]):
    ds = {}

    for dir in dirs:
        d = df[~df[f'{dir}epoch'].isna()]
        d = d[[c for c in d.columns if dir in c]]

        ds[dir] = d

    return ds

def load_and_extract(api, run_name):
    run = api.run(run_name)
    df = run.history(samples=100000)
    # print([f for f in list(df.columns) if "checkpoint" not in f])
    ds = extract_stats(df)
    return ds

In [173]:
def extract_metrics_at_epoch(
    dataframe,
    epoch,
    wandb_dirs=[
        "map_metrics/",
        "laplace_LORAXS_KRON/",
        "laplace_LORAXS_DIAG/",
        "laplace/base_",
    ],
):

    metrics_selected = {}
    for prefix_dir in wandb_dirs:
        try:
            dir_results = dataframe[prefix_dir]
            dir_metrics = dir_results[dir_results[prefix_dir + "epoch"] == epoch]  # Filter by epoch
            dir_metrics = dir_metrics.to_dict(orient='records')[0]

            metrics_selected.update(dir_metrics)
        except Exception as e:
            print(f"Error processing {prefix_dir}: {e}")

    return metrics_selected

In [174]:
CHECKPOINT_EPOCH = 20

In [175]:
KEEP_RUNS_WITH_PREFIXES = ["eap_", "eap2_", "eap3_", "eapm_"]

In [176]:
cfgs = []
for run in runs:
    cfg = extract_wandb_params(json.loads(run.json_config))

    if not any([run.name.startswith(pref) for pref in KEEP_RUNS_WITH_PREFIXES]):
        print(f"Skipping run: {run.name}")
        continue

    print(f"Parsing run: {run.name}")

    cfg["name"] = run.name
    cfg["state"] = run.state
    cfg["id"] = prj+"/"+run.id

    try:
        stats = load_and_extract(api, prj+"/"+run.id)
        metrics_at_epoch = extract_metrics_at_epoch(stats, epoch=CHECKPOINT_EPOCH)
        cfg.update(metrics_at_epoch)
    except Exception as e:
        print(f"  Failed to load stats from {run.name}", e)
        continue

    cfgs.append(cfg)
    

Skipping run: exp_roberta_on_cola_svd_test roberta-large_cola_loraxs_seed3407_lr0.001_cls_lr0.005_ep5
Skipping run: exp_roberta_on_cola_svd_longrun roberta-large_cola_loraxs_seed3407_lr0.001_cls_lr0.005_ep201
Skipping run: exp_roberta_seed1410_on_cola_svd_longrun roberta-large_cola_loraxs_seed1410_lr0.001_cls_lr0.005_ep201
Skipping run: exp_roberta_on_mrpc_svd_longrun roberta-large_mrpc_loraxs_seed3407_lr0.001_cls_lr0.005_ep201
Skipping run: exp_roberta_on_rte_svd_longrun roberta-large_rte_loraxs_seed3407_lr0.001_cls_lr0.005_ep201
Parsing run: eap_8_20260115_roberta_cola_CLASSIFIER_WEIGHT_DECAY_0_01_CLS_LEARNING_RATE_0_005_LEARNING_RATE_0_001_LORA_ALPHA_16_LORA_DROPOUT_0_0_LORA_R_8_LORA_WEIGHT_DECAY_0_01_RECONSTRUCT_TYPE_random-1_2_svd_SEED_3407 roberta-large_cola_loraxs_seed3407_lr0.001_cls_lr0.005_ep31
Parsing run: eap_3_20260115_roberta_cola_CLASSIFIER_WEIGHT_DECAY_0_01_CLS_LEARNING_RATE_0_005_LEARNING_RATE_0_001_LORA_ALPHA_16_LORA_DROPOUT_0_0_LORA_R_8_LORA_WEIGHT_DECAY_0_01_RECONST

In [177]:
df = pd.DataFrame(cfgs)

In [179]:
df = df.groupby(["task_name", "reconstruction_type"]).agg({'id':'count',
                                                     'laplace_LORAXS_KRON/eval_nll': 'mean',
                                                     'laplace_LORAXS_DIAG/eval_nll': 'mean',
                                                     'laplace/base_eval_nll': 'mean',
                                                     'laplace_LORAXS_KRON/eval_acc': 'mean',
                                                     'laplace_LORAXS_DIAG/eval_acc': 'mean',
                                                     'laplace/base_eval_acc': 'mean',
                                                     'laplace_LORAXS_KRON/eval_ece': 'mean',
                                                     'laplace_LORAXS_DIAG/eval_ece': 'mean',
                                                     'laplace/base_eval_ece': 'mean',
                                                     }).reset_index()

In [180]:
df_cola = df[df["task_name"]=="cola"]

In [181]:
df_cola.sort_values(by="laplace_LORAXS_KRON/eval_acc", ascending=False)

,task_name,reconstruction_type,id,laplace_LORAXS_KRON/eval_nll,laplace_LORAXS_DIAG/eval_nll,laplace/base_eval_nll,laplace_LORAXS_KRON/eval_acc,laplace_LORAXS_DIAG/eval_acc,laplace/base_eval_acc,laplace_LORAXS_KRON/eval_ece,laplace_LORAXS_DIAG/eval_ece,laplace/base_eval_ece
8,cola,svd,4,0.389805,0.385808,0.431690,0.841563,0.841803,0.841802,0.055966,0.048585,0.082059
0,cola,cca,4,0.381463,0.375961,0.413000,0.840844,0.840844,0.840604,0.046627,0.042313,0.069943
3,cola,dct-1/2_svd,4,0.416069,0.409647,0.453869,0.824065,0.823826,0.824065,0.066581,0.061505,0.089793
5,cola,finder-1/2_svd,4,0.461393,0.451491,0.514678,0.803212,0.803691,0.802733,0.073489,0.066908,0.096050
7,cola,random-1/2_svd,4,0.456403,0.447956,0.513881,0.797939,0.798178,0.798178,0.082886,0.075593,0.105900
4,cola,finder,4,0.552788,0.528956,0.577712,0.788111,0.788431,0.788431,0.114530,0.099878,0.124087
2,cola,dct,4,0.519266,0.510004,0.532457,0.786913,0.786913,0.786913,0.099636,0.094069,0.106743
1,cola,cca-1/2_svd,4,0.530747,0.524144,0.555138,0.748322,0.748562,0.748322,0.058351,0.053398,0.074476
6,cola,random,4,0.622536,0.618691,0.629948,0.731863,0.731863,0.731863,0.149043,0.144677,0.151778


In [182]:
df_mrpc = df[df["task_name"]=="mrpc"]

In [183]:
df_mrpc.sort_values(by="laplace_LORAXS_KRON/eval_acc", ascending=False)

,task_name,reconstruction_type,id,laplace_LORAXS_KRON/eval_nll,laplace_LORAXS_DIAG/eval_nll,laplace/base_eval_nll,laplace_LORAXS_KRON/eval_acc,laplace_LORAXS_DIAG/eval_acc,laplace/base_eval_acc,laplace_LORAXS_KRON/eval_ece,laplace_LORAXS_DIAG/eval_ece,laplace/base_eval_ece
13,mrpc,finder,3,0.323042,0.328835,0.404334,0.863971,0.865196,0.863971,0.040324,0.055629,0.083336
14,mrpc,finder-1/2_svd,3,0.325677,0.333450,0.384441,0.863562,0.864379,0.863562,0.046740,0.060945,0.080523
11,mrpc,dct,3,0.324909,0.331797,0.410783,0.861111,0.861928,0.861111,0.043590,0.052000,0.080897
12,mrpc,dct-1/2_svd,3,0.324909,0.331797,0.410783,0.861111,0.861928,0.861111,0.043590,0.052000,0.080897
17,mrpc,svd,3,0.324909,0.331797,0.410783,0.861111,0.861928,0.861111,0.043590,0.052000,0.080897
9,mrpc,cca,3,0.322932,0.330672,0.391577,0.857843,0.857026,0.857026,0.047251,0.064831,0.080552
10,mrpc,cca-1/2_svd,3,0.322932,0.330672,0.391577,0.857843,0.857026,0.857026,0.047251,0.064831,0.080552
16,mrpc,random-1/2_svd,3,0.323981,0.327664,0.429441,0.856209,0.854575,0.855392,0.051265,0.044106,0.089544
15,mrpc,random,3,0.336861,0.339243,0.464262,0.849673,0.848039,0.848856,0.048177,0.041295,0.103067
